# CLIP

## 概述

🧩 1.1 CLIP要解决的问题

在CLIP之前，计算机视觉的模型（ResNet、ViT）主要依赖 人工标注的分类数据集（如ImageNet），标签空间有限且昂贵。
而互联网上有大量的 图文对（image–text pairs），比如微博、Instagram、Reddit等。

👉 CLIP的想法：

我能不能直接用图文对作为训练数据？
让模型学会“图片和文字是否匹配”，而不是强行做分类。

🧠 1.2 CLIP的核心思想

CLIP不是分类模型，而是一个 图像–文本对齐模型。

它训练两个编码器：

一个将图片映射为向量： image_encoder(image) → v_i

一个将文字映射为向量： text_encoder(text) → v_t

训练目标是让：

匹配的图文对 (v_i, v_t) 余弦相似度高；

不匹配的图文对相似度低。

最终效果是：

模型理解“图像和文本之间的语义关系”。

## CLIP的模型结构

CLIP由两部分组成：

| 模块 | 说明 | 常见实现 |
| :---: | :---: | :---: |
|图像编码器 (Image Encoder)|把图片编码为向量|ResNet / Vision Transformer|
|文本编码器 (Text Encoder)|把文字编码为向量|Transformer Encoder|

🧩 结构示意

图片 → Image Encoder → 图像特征向量

文字 → Text Encoder → 文本特征向量

两者进入 → 对比学习（计算相似度矩阵）

📚 关键细节

向量都通过 L2 normalization 标准化。

相似度用 cosine similarity。

损失是 InfoNCE / 对比学习损失。

## 损失函数（CLIP的灵魂）

### 图文对比学习任务描述

假设有一批次的图文对：  
$$
\{(I_1, T_1), (I_2, T_2), \dots, (I_N, T_N)\}
$$

分别得到图像特征和文本特征：  
$$
V_I = [v_1, v_2, \dots, v_N], \quad V_T = [t_1, t_2, \dots, t_N]
$$


### 相似度计算
计算相似度矩阵（带温度系数 `temperature`）：  
$$
S_{ij} = \text{cosine}(v_i, t_j) \times \text{temperature}
$$

**目标**：  
- 对角线（$i = j$）的匹配得分最高；  
- 其余（$i \neq j$）的错配得分低。


### 损失函数
对称的交叉熵损失：
```
image_loss = cross_entropy(similarity_matrix, target=正确匹配索引)
text_loss = cross_entropy(similarity_matrix.T, target=正确匹配索引)
loss = (image_loss + text_loss) / 2 # 平均损失
```

正确的匹配索引就是对角线！(1, 2, 3, 4, ...)

### Mini-CLIP

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel

class CLIPModel(nn.Module):
    def __init__(self, image_encoder, text_encoder, embed_dim=512):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        self.image_proj = nn.Linear(image_encoder.embed_dim, embed_dim)
        self.text_proj = nn.Linear(text_encoder.config.hidden_size, embed_dim)
        self.temperature = nn.Parameter(torch.tensor(1.0))
        
    def forward(self, image, text_input_ids, text_attention_mask):
        image_features = self.image_proj(self.image_encoder(image))
        text_features = self.text_proj(self.text_encoder(text_input_ids, attention_mask=text_attention_mask).pooler_output)
        
        image_features = F.normalize(image_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)
        
        logits = torch.matmul(image_features, text_features.t()) * torch.exp(self.temperature)
        labels = torch.arange(len(logits), device=logits.device)
        loss_i = F.cross_entropy(logits, labels)
        loss_t = F.cross_entropy(logits.T, labels)
        loss = (loss_i + loss_t) / 2
        return loss, logits

## CLIP 实现零样本分类 

In [3]:
import torch
from PIL import Image
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
import os

# 设置镜像源（阿里云）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 加载预训练的CLIP (OPENAI 的ViT-B/32)
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

image_paths = ['./images/bottle.jpg', './images/pen.jpg', './images/keys.jpg']
images = [Image.open(p).convert("RGB") for p in image_paths]

labels = ["a photo of a bottle", "a photo of a pen", "a photo of keys"]

inputs = processor(text=labels, images=images, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=-1)
    
for i, path in enumerate(image_paths):
    print(f"\nImage:{path}")
    for j, label in enumerate(labels):
        print(f"{label:<25}:{probs[i, j]:.4f}")
    print(f" Prediction: {labels[probs[i].argmax()]}")


Image:./images/bottle.jpg
a photo of a bottle      :0.9984
a photo of a pen         :0.0015
a photo of keys          :0.0001
 Prediction: a photo of a bottle

Image:./images/pen.jpg
a photo of a bottle      :0.0000
a photo of a pen         :1.0000
a photo of keys          :0.0000
 Prediction: a photo of a pen

Image:./images/keys.jpg
a photo of a bottle      :0.0019
a photo of a pen         :0.0099
a photo of keys          :0.9882
 Prediction: a photo of keys
